# Melbourne House Prices - Exploratory Data Analysis

**Goal:** understand the data before building a model. What drives price? What's missing? What looks wrong?

Dataset: Melbourne Housing Snapshot - 13,580 property sales from 2016-2017.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

df = pd.read_csv("../data/raw/melb_data.csv", parse_dates=["Date"], dayfirst=True)
print(df.shape)
df.head()

## 1. What's in each column?
Check the data types. Numbers should be numbers, text should be text.

In [ ]:
df.info()

## 2. Missing values
A model can't learn from blanks, so we need to know how much is missing and decide what to do about it.

In [ ]:
missing = df.isna().mean().mul(100).sort_values(ascending=False)
missing[missing > 0].to_frame("% missing")

**Note for later:** if a column is missing in almost half the rows, filling it in with guesses could add more noise than signal. We'll decide per column in the modelling notebook.

## 3. The target: Price
Price is what we want to predict. Let's see its shape.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["Price"], bins=60, ax=axes[0])
axes[0].set_title("Price")
sns.histplot(np.log(df["Price"]), bins=60, ax=axes[1])
axes[1].set_title("log(Price)")
plt.tight_layout()
plt.show()

print("Median price:", f"${df['Price'].median():,.0f}")
print("Mean price:  ", f"${df['Price'].mean():,.0f}")

**Why log?** Prices have a long tail - a few very expensive houses stretch the chart to the right (the mean ends up above the median). Taking the log squashes that tail into a nicer bell shape, which helps many models. Think of it as measuring price in "how many times bigger" instead of "how many dollars bigger".

## 4. Price by property type
`h` = house, `t` = townhouse, `u` = unit/apartment.

In [ ]:
print(df.groupby("Type")["Price"].agg(["count", "median"]))

sns.boxplot(data=df, x="Type", y="Price", showfliers=False)
plt.title("Price by property type (outliers hidden)")
plt.show()

## 5. Price vs distance to the CBD

In [ ]:
sns.scatterplot(data=df, x="Distance", y="Price", hue="Type", alpha=0.3, s=10)
plt.yscale("log")
plt.title("Price vs distance from CBD (log scale)")
plt.show()

print("Correlation (Distance vs log Price):", round(df["Distance"].corr(np.log(df["Price"])), 2))

## 6. Price by number of rooms

In [ ]:
sns.boxplot(data=df[df["Rooms"] <= 6], x="Rooms", y="Price", showfliers=False)
plt.title("Price by number of rooms (1-6)")
plt.show()

## 7. Price by region

In [ ]:
region = df.groupby("Regionname")["Price"].median().sort_values()
region.plot(kind="barh", figsize=(8, 4), title="Median price by region")
plt.xlabel("Median price ($)")
plt.show()

## 8. How do the numeric columns relate to price?
Correlation runs from -1 to 1. Near 0 = no straight-line relationship.

In [ ]:
num_cols = ["Price", "Rooms", "Bedroom2", "Bathroom", "Car", "Distance",
            "Landsize", "BuildingArea", "YearBuilt", "Propertycount"]
corr = df[num_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation between numeric columns")
plt.show()

## 9. Values that look wrong
Real data has mistakes. Let's hunt for impossible values.

In [ ]:
checks = {
    "Landsize = 0": (df["Landsize"] == 0).sum(),
    "BuildingArea = 0": (df["BuildingArea"] == 0).sum(),
    "YearBuilt before 1850": (df["YearBuilt"] < 1850).sum(),
    "Bedroom2 = 0 but Rooms > 0": ((df["Bedroom2"] == 0) & (df["Rooms"] > 0)).sum(),
}
pd.Series(checks).to_frame("rows")

In [ ]:
# Rooms vs Bedroom2 - are they basically the same thing?
print("Rooms == Bedroom2 in", f"{(df['Rooms'] == df['Bedroom2']).mean():.0%}", "of rows")

## Takeaways
Fill these in from your own results - you need to be able to say them in an interview.

1. **Price shape:** ...
2. **Biggest drivers:** ...
3. **Data problems to fix before modelling:** ...